# 02. LangGraph 설계 원칙

> 그래프를 어떻게 쪼개야 디버깅·재실행·HITL이 쉬워질까요? 5단계 설계 방법론과 노드 분해 원칙을 따라가며 흔한 안티패턴을 미리 차단해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **5단계 구현 방법론**(워크플로우 매핑 → 타입 식별 → 상태 설계 → 노드 구현 → 엣지 연결)을 적용해서 LangGraph 앱을 체계적으로 설계할 수 있어요
2. **노드 분해 원칙**(단일 책임)을 이해하고, 스트리밍·체크포인트·디버깅에 유리한 구조로 그래프를 설계할 수 있어요
3. **4가지 에러 유형**(Transient / LLM-recoverable / User-fixable / Unexpected)별로 적절한 처리 전략을 선택할 수 있어요
4. **안티패턴**(상태에 포맷된 텍스트 저장, 거대 단일 노드, @task 없는 사이드 이펙트 등)을 식별하고 올바른 패턴으로 리팩터링할 수 있어요

## 사전 지식

- Part 02에서 배운 `StateGraph`, `Node`, `Edge`, `START/END`, `add_messages` reducer
- Part 03-01에서 배운 7가지 패턴(Augmented LLM, Prompt Chaining, Parallelization 등)
- `InMemorySaver`, `interrupt()`, `Command` 기본 사용법 (Part 02-07, 02-08)


## LangGraph 설계 원칙이란?

LangGraph로 에이전트를 만들 때 가장 큰 질문은 **"어떻게 설계해야 하는가?"** 예요.
코드를 바로 쓰기 전에 몇 가지 핵심 원칙을 이해하면 유지보수하기 쉽고 안정적인 그래프를 만들 수 있어요.

### 왜 설계 원칙이 중요한가요?

집을 짓는다고 상상해보세요. 설계도 없이 벽돌부터 쌓으면 나중에 벽을 허물고 다시 지어야 해요. 소프트웨어도 마찬가지예요. **5분의 설계가 5시간의 디버깅을 절약**해요. 특히 LangGraph는 노드 분해, 상태 설계, 에러 처리 등 의사결정이 많아서 체계적인 접근이 더욱 중요해요.

이 노트북에서는 **EmailAgent**라는 예제를 통해 설계 원칙을 단계적으로 살펴볼게요.
EmailAgent는 이메일을 받아 분류하고, 관련 정보를 검색하고, 사람 검토 후 답장을 전송하는 시스템이에요.

```mermaid
flowchart TD
    A(["시작"]) --> B["이메일 분류<br/>classify_email"]
    B --> C{"분류 결과"}
    C -->|"중요"| D["컨텍스트 검색<br/>retrieve_context"]
    C -->|"스팸"| Z(["종료"])
    D --> E["초안 작성<br/>draft_response"]
    E --> F["인간 검토<br/>human_review"]
    F -->|"승인"| G["이메일 전송<br/>send_email"]
    F -->|"수정 요청"| E
    G --> Z

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef decision fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef human fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A input
    class B,D,E,G process
    class C,F decision
    class Z output
```

| 설계 원칙 | 핵심 내용 | 비유 |
|-----------|----------|------|
| **5단계 방법론** | 워크플로우 매핑 -> 타입 식별 -> 상태 설계 -> 노드 구현 -> 엣지 연결 | 건축 설계도 그리기 |
| **노드 분해** | 단일 책임 원칙, 스트리밍/체크포인트/디버깅에 최적화 | 레고 블록처럼 쪼개기 |
| **상태 설계** | raw data 저장, formatted text 금지, reducer 선택 | 데이터베이스 스키마 설계 |
| **에러 처리** | 4가지 유형별 전략 (Transient/LLM-recoverable/User-fixable/Unexpected) | 병원 응급실 분류 (트리아지) |
| **안티패턴** | 피해야 할 패턴과 올바른 대안 | 흔한 실수 목록 |

> 🔑 **핵심 개념**: 좋은 LangGraph 설계는 **"그래프를 어떻게 분해할 것인가"** 에서 출발해요. 노드를 너무 크게 만들면 디버깅이 어렵고, 너무 잘게 쪼개면 복잡해져요. 각 노드가 **하나의 명확한 책임**을 갖도록 설계하는 것이 핵심이에요.


## 환경 설정


In [ ]:
# 환경 변수 로드 (.env 파일에서 API 키를 읽어와요)
from dotenv import load_dotenv
load_dotenv()


In [ ]:
# LangSmith 추적 설정 (학습 중 그래프 실행 과정을 추적할 수 있어요)
import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Part03-DesignPrinciples"


In [ ]:
# LangChain V1 모델 초기화
from langchain.chat_models import init_chat_model

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 다른 모델 옵션:
#   - Anthropic: "anthropic:claude-sonnet-4-5"
#   - Google Gemini: "google_genai:gemini-2.0-flash"
model = init_chat_model("openai:gpt-4o-mini")
# 모델 초기화 완료


## 1. 5단계 구현 방법론

LangGraph 앱을 만들 때 **체계적인 순서**를 따르면 설계 실수를 줄일 수 있어요.
아래 5단계를 EmailAgent 예제에 적용해볼게요.

```mermaid
flowchart LR
    S1["1단계<br>워크플로우 매핑"] --> S2["2단계<br>타입 식별"]
    S2 --> S3["3단계<br>상태 설계"]
    S3 --> S4["4단계<br>노드 구현"]
    S4 --> S5["5단계<br>엣지 연결"]

    classDef step fill:#cce5ff,stroke:#007bff,color:#004085
    class S1,S2,S3,S4,S5 step
```

### 1단계: 워크플로우 매핑

먼저 시스템이 해야 할 **작업의 흐름**을 글로 정리해요.
- 이메일을 받아서 스팸인지 중요 이메일인지 분류해요
- 중요 이메일이면 관련 문서를 검색해서 컨텍스트를 구성해요
- 검색 결과를 바탕으로 답장 초안을 작성해요
- 사람이 검토하고 승인하거나 수정을 요청해요
- 승인된 초안을 이메일로 전송해요

### 2단계: 타입 식별

각 단계에서 **어떤 데이터**가 흐르는지 파악해요.
- 입력: 이메일 원문 (str)
- 분류 결과: "important" 또는 "spam" (str)
- 검색된 컨텍스트: 문서 목록 (list[str])
- 초안: 답장 텍스트 (str)
- 인간 피드백: 승인 여부 + 수정 의견 (bool + str)


### 3단계: 상태 설계

상태(State)는 그래프 전체에서 공유되는 **메모리**예요. 설계 시 두 가지 핵심 규칙이 있어요.

**규칙 1: raw data를 저장하세요. formatted text는 금지예요.**

```python
# 나쁜 예 - 포맷된 텍스트를 상태에 저장하면 안 돼요
class BadState(TypedDict):
    email_summary: str  # "발신자: kim@example.com\n제목: 안녕하세요\n내용: ..."

# 좋은 예 - 구조화된 raw data를 저장해요
class GoodState(TypedDict):
    email_sender: str    # "kim@example.com"
    email_subject: str   # "안녕하세요"
    email_body: str      # "..."
```

**규칙 2: 리스트 필드는 reducer를 명시하세요.**

- 메시지 히스토리처럼 **누적**해야 하면 `Annotated[list, operator.add]`
- 매번 **덮어써야** 하면 reducer 없이 `list[str]`


In [ ]:
from typing import TypedDict, Optional

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3단계: EmailAgent 상태 설계
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4단계: 노드 구현

각 노드는 **단일 책임 원칙**을 따라야 해요. 하나의 노드가 여러 가지 일을 동시에 하면:
- 어느 부분에서 에러가 났는지 파악하기 어려워요
- 특정 단계만 스트리밍으로 확인하기 어려워요
- 체크포인트 저장 시 중간 상태를 활용하기 어려워요


In [ ]:
from langchain.messages import HumanMessage, SystemMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 4단계: EmailAgent 노드 구현
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5단계: 엣지 연결

마지막으로 노드들을 연결해서 그래프를 완성해요.
조건에 따라 다른 경로로 가야 할 때는 `add_conditional_edges`를 사용해요.


In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 5단계: 엣지 연결 - 전체 EmailAgent 그래프 조립
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → classify_email → (important) retrieve_context → draft_response → human_rev
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import uuid

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: EmailAgent 실행: 스팸 이메일 처리 (중단 없이 바로 종료)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.types import Command

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: EmailAgent 실행: 중요 이메일 처리 (인간 검토 포함)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: EmailAgent 재개: Command(resume=...)로 인간 검토 결과 전달
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. 에러 처리 전략

### 왜 에러 유형을 구분해야 하나요?

병원 응급실에서 모든 환자를 같은 방식으로 치료하지 않는 것처럼, 에러도 **유형에 따라 다른 전략**이 필요해요. 네트워크가 잠깐 끊긴 거라면 재시도하면 되지만, 사용자가 입력을 잘못한 거라면 재시도해도 똑같이 실패해요.

LangGraph에서 에러는 4가지 유형으로 나뉘어요. 각 유형에 맞는 처리 전략이 달라요.

```mermaid
flowchart TD
    ERR["에러 발생"] --> T1["Transient<br/>일시적 오류"]
    ERR --> T2["LLM-recoverable<br/>LLM으로 복구 가능"]
    ERR --> T3["User-fixable<br/>사용자가 수정 가능"]
    ERR --> T4["Unexpected<br/>예상치 못한 오류"]

    T1 --> S1["RetryPolicy로<br/>자동 재시도"]
    T2 --> S2["오류 피드백을 상태에 저장<br/>-> LLM 노드로 루프백"]
    T3 --> S3["interrupt()로 중단<br/>-> 사용자 입력 대기"]
    T4 --> S4["예외를 그대로 전파<br/>-> 상위에서 처리"]

    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef strategy fill:#d4edda,stroke:#28a745,color:#155724

    class ERR,T1,T2,T3,T4 error
    class S1,S2,S3,S4 strategy
```

| 에러 유형 | 예시 | 처리 전략 | 비유 |
|-----------|------|-----------|------|
| **Transient** | API 타임아웃, 일시적 네트워크 오류 | `RetryPolicy` 자동 재시도 | 전화가 안 터져서 재다이얼 |
| **LLM-recoverable** | 잘못된 JSON 형식, 형식 오류 | 오류를 상태에 담아 LLM 노드로 루프백 | 보고서 양식이 틀려서 다시 작성 요청 |
| **User-fixable** | 모호한 입력, 권한 부족 | `interrupt()`로 사용자에게 재입력 요청 | 서류 미비로 창구 재방문 요청 |
| **Unexpected** | 코드 버그, 데이터 오염 | 예외 전파, 상위에서 처리 | 원인 불명 고장 -> 전문가 호출 |

> 🔑 **핵심 개념**: 에러를 잡아서 무조건 감추는 것은 나쁜 패턴이에요. **각 에러 유형에 맞는 전략**을 쓰면 시스템이 더 강건(robust)해요. Unexpected 에러는 숨기지 말고 전파해서 개발자가 알 수 있게 해야 해요.


In [ ]:
from langgraph.types import RetryPolicy
from langgraph.graph import StateGraph
from typing import TypedDict

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에러 유형 1: Transient - RetryPolicy로 자동 재시도
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → call_api → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import json

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에러 유형 2: LLM-recoverable - 오류 피드백 루프백
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → parse → validate → parse(루프) 또는 END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. Command: 상태 업데이트 + 라우팅 동시 처리

`Command`는 노드에서 **상태를 업데이트하면서 동시에 다음 노드를 지정**할 수 있는 강력한 기능이에요.
일반 `add_conditional_edges`와 달리, 노드 내부에서 다음 이동 경로를 동적으로 결정할 수 있어요.

```mermaid
flowchart LR
    N1["노드 A"] -->|"Command(update=..., goto='B')"| N2["노드 B"]
    N1 -->|"Command(update=..., goto='C')"| N3["노드 C"]

    classDef node fill:#cce5ff,stroke:#007bff,color:#004085
    class N1,N2,N3 node
```


In [ ]:
from langgraph.types import Command
from typing import Union

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Command: 상태 업데이트 + 라우팅을 한 번에 처리하는 패턴
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Command 패턴 실행 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 안티패턴과 올바른 패턴

실무에서 자주 보이는 잘못된 패턴과 올바른 대안을 살펴볼게요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 안티패턴 1: 상태에 포맷된 텍스트 저장
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 안티패턴 2: 거대 단일 노드 (Monolithic Node)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → extract_info → validate → format_output → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. interrupt() 사용 규칙

`interrupt()`는 그래프 실행을 일시 중단하고 외부 입력을 기다려요.
올바르게 사용하려면 **3가지 규칙**을 지켜야 해요.

| 규칙 | 올바른 사용 | 잘못된 사용 |
|------|------------|------------|
| **위치** | 노드의 **처음**에 배치 | 노드 중간 또는 끝에 배치 |
| **예외 처리** | try/except **바깥**에 위치 | try/except **안**에 위치 |
| **순서 일관성** | resume 시 같은 순서 보장 | resume 시 다른 조건으로 건너뜀 |


In [ ]:
from langgraph.types import interrupt, Command

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: interrupt() 올바른 사용 패턴 예시
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: interrupt() + Command(resume=) 실행 흐름
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 종합 실습: 설계 원칙 적용

지금까지 배운 설계 원칙을 종합해서 간단한 콘텐츠 검수 시스템을 만들어볼게요.


In [ ]:
# ============================================================
# TODO: 콘텐츠 검수 시스템 설계 실습
#
# 아래 요구사항에 맞게 LangGraph 앱을 설계해보세요.
#
# 요구사항:
# 1. 사용자가 작성한 블로그 포스트(제목 + 본문)를 받아요
# 2. LLM이 포스트를 검토해서 '통과(pass)' 또는 '수정필요(revise)' 판정을 해요
# 3. '수정필요'이면 구체적인 개선 제안을 생성해요
# 4. 편집자(interrupt)에게 최종 승인을 받아요
# 5. 승인되면 "발행 완료" 메시지를 출력해요
#
# 설계 힌트:
# - 5단계 방법론을 따라요: 워크플로우 매핑 → 타입 식별 → 상태 설계 → 노드 구현 → 엣지 연결
# - 상태에는 raw data를 저장하세요 (포맷된 텍스트 금지)
# - 단일 책임 원칙: 하나의 노드에 너무 많은 역할을 주지 마세요
# - interrupt()는 노드의 처음에 배치하세요
#
# 예상 결과:
# - 그래프가 시각화되고
# - 스팸 필터 통과 → LLM 검토 → interrupt 발생
# - Command(resume="yes")로 재개 → 발행 완료 출력
# ============================================================

# 1단계: 상태 설계
# class ContentReviewState(TypedDict):
#     title: str
#     content: str
#     review_verdict: str      # "pass" 또는 "revise"
#     improvement_suggestions: str
#     editor_approved: bool
#     published: bool

# 2단계 이후: 직접 구현해보세요!

# 설계 원칙을 적용해서 콘텐츠 검수 시스템을 만들어보세요!
# 힌트: 5단계 방법론을 따라 단계적으로 구현해요


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **5단계 방법론**: 워크플로우 매핑 → 타입 식별 → 상태 설계 → 노드 구현 → 엣지 연결 순서로 체계적으로 설계해요
- **상태 설계 원칙**: raw data를 저장하고 포맷된 텍스트는 금지해요. 리스트 필드는 reducer를 명시해요
- **노드 분해**: 단일 책임 원칙으로 노드를 작게 유지하면 스트리밍·체크포인트·디버깅이 쉬워져요
- **4가지 에러 유형**: Transient(RetryPolicy) / LLM-recoverable(루프백) / User-fixable(interrupt) / Unexpected(전파)
- **Command**: 상태 업데이트와 라우팅을 한 번에 처리할 때 사용해요
- **interrupt() 규칙**: 노드 처음에 배치, try/except 바깥에 위치, 순서 일관성 유지
- **안티패턴**: 포맷된 텍스트를 상태에 저장하거나, 하나의 노드가 너무 많은 역할을 하면 안 돼요


## 다음 노트북 예고

다음 `03-Reliability-Engineering.ipynb`에서는 **신뢰성 엔지니어링 5원칙 — Constrain · Inform · Verify · Correct · HITL**을 배워요. "에이전트가 실수했다"를 모델 탓이 아니라 주변 코드를 한 겹 덧댈 신호로 해석하는 사고틀이에요. 이후 04~13장에서 배울 middleware, memory, guardrails, testing이 모두 이 5가지 도구의 구체화예요.
